# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [4]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /opt/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /opt/bigdata/big-data-postgraduate
Salidas: /opt/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [17]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.
# Mostrar los tipos y los valores que no pudieron convertirse.

# Código para el bloque de desarrollo de la Actividad 1
import pandas as pd

# 1. Asegurar la asignación sobre la variable global 'eva'
eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip().str.zfill(2)
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(
    eva["Área cosechada"].str.replace(",", ".", regex=False).str.strip(), errors="coerce"
)
eva["produccion_t"] = pd.to_numeric(
    eva["Producción"].str.replace(",", ".", regex=False).str.strip(), errors="coerce"
)

# 2. Verificación de nulos/no convertidos
print("Ausencias en columnas preparadas:")
print("codigo_departamento NA:", eva["codigo_departamento"].isna().sum())
print("anio NA:", eva["anio"].isna().sum())
print("area_ha NA:", eva["area_ha"].isna().sum())
print("produccion_t NA:", eva["produccion_t"].isna().sum())

# 3. Vista previa de la estructura preparada
display(
    eva[
        [
            "codigo_departamento",
            "Departamento",
            "Cultivo",
            "anio",
            "area_ha",
            "produccion_t",
        ]
    ].head()
)

Ausencias en columnas preparadas:
codigo_departamento NA: 0
anio NA: 0
area_ha NA: 0
produccion_t NA: 0


,codigo_departamento,Departamento,Cultivo,anio,area_ha,produccion_t
0,05,Antioquia,Apio,2021,8.0,128.00
1,05,Antioquia,Apio,2021,8.0,152.00
2,05,Antioquia,Apio,2022,8.0,127.44
3,05,Antioquia,Apio,2022,8.0,152.00
4,05,Antioquia,Apio,2023,7.0,133.00


**Interpretación del resultado:**

Los conjuntos de datos abiertos suelen conservar la nomenclatura institucional con tildes y mayúsculas (Código Dane departamento, Área cosechada). Mapear explícitamente estos campos garantiza que la extracción y el tipado se realicen sin fallos de índice, permitiendo estandarizar los códigos territoriales a dos dígitos (por ejemplo, asegurando que Boyacá quede como "15") para los filtros analíticos posteriores.

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [18]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
# Mostrar los conteos de entrada, selección y diferencia.

# Código para el bloque de desarrollo de la Actividad 2
mask = (
    eva["codigo_departamento"].eq("15") &
    eva["anio"].isin([2023, 2024]) &
    eva["area_ha"].gt(0) &
    eva["produccion_t"].ge(0)
)

consulta_loc = eva.loc[mask].copy()

total_original = len(eva)
total_seleccion = len(consulta_loc)
diferencia = total_original - total_seleccion

print(f"Total original:    {total_original:,}")
print(f"Total seleccionado: {total_seleccion:,}")
print(f"Diferencia:        {diferencia:,}")

Total original:    166,732
Total seleccionado: 4,924
Diferencia:        161,808


**Interpretación del resultado:**

Al aislar el código "15" (Boyacá) en los periodos recientes (2023 y 2024) y exigir área cosechada mayor a cero, se eliminan registros inactivos o sin producción reportada, centrando el análisis exclusivamente en el desempeño agrícola real del altiplano cundiboyacense.

De las 166.732 observaciones de la base nacional de las Evaluaciones Agropecuarias Municipales (EVA), 4.924 corresponden al filtro territorial de Boyacá en el bienio 2023-2024 con actividad productiva efectiva (área cosechada $>0$). La diferencia de 161.808 registros corresponde a reportes de otros departamentos, años anteriores o registros inactivos/sin cosecha.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [20]:
# Desarrollo de la actividad 3
# Construir consulta_query.
# Comprobar que consulta_loc y consulta_query son equivalentes.

# Código para el bloque de desarrollo de la Actividad 3
codigo = "15"

consulta_query = eva.query(
    "codigo_departamento == @codigo and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0",
    engine="python"
).copy()

# Comprobar equivalencia exacta
pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("Comprobación exitosa: consulta_loc y consulta_query son exactamente iguales.")

Comprobación exitosa: consulta_loc y consulta_query son exactamente iguales.


**Interpretación del resultado:**

La verificación mediante pd.testing.assert_frame_equal confirma que la máscara de álgebra booleana en .loc y el motor de evaluación de .query producen arreglos idénticos en filas, columnas, tipos de datos e índices. El método .query ofrece mayor legibilidad narrativa en scripts extensos, mientras que .loc facilita un control explícito sobre las máscaras lógicas intermedias.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [23]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
# Verificar los totales y los porcentajes por año.

# Código para el bloque de desarrollo de la Actividad 4
resumen = (
    consulta_loc.groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

# Total de registros por año (vía transform)
resumen["total_anio"] = resumen.groupby("anio")["registros"].transform("sum")

# Porcentaje anual
resumen["porcentaje_anual"] = (resumen["registros"] / resumen["total_anio"]) * 100

# Ordenar por año ascendente y volumen de registros descendente
resumen = resumen.sort_values(by=["anio", "registros"], ascending=[True, False]).reset_index(drop=True)

# Verificaciones
assert resumen["registros"].sum() == len(consulta_loc), "La suma de registros no coincide con consulta_loc"
print(f"Suma total de registros verificada: {resumen['registros'].sum():,} filas.")

display(resumen.head(10))

Suma total de registros verificada: 4,924 filas.


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.811166
1,2023,Maíz,194,2436,7.963875
2,2023,Arveja,176,2436,7.224959
3,2023,Frijol,162,2436,6.650246
4,2023,Tomate,113,2436,4.638752
5,2023,Cebolla de bulbo,93,2436,3.817734
6,2023,Yuca,82,2436,3.366174
7,2023,Caña,63,2436,2.586207
8,2023,Trigo,58,2436,2.380952
9,2023,Mora,51,2436,2.093596


**Interpretación del resultado:**

El número de reportes en el departamento de Boyacá se mantiene estable entre 2023 (2.436 registros) y 2024 (2.488 registros). Los cultivos tradicionales como la Papa, el Maíz, la Arveja y el Fríjol lideran la frecuencia de observaciones agrícolas en las Evaluaciones Agropecuarias.

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [25]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
# Exportar resumen y explicar dos resultados en la celda siguiente.

tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    fill_value=0
)

# Exportar resultado a kit/salidas/soluciones/PD02/resumen_eva.csv
resumen.to_csv(OUT / "resumen_eva.csv", index=False, encoding="utf-8-sig")

print("Tabla dinámica de registros por cultivo y año (Top 15 por registros totales):")
display(tabla_dinamica.assign(total=tabla_dinamica.sum(axis=1)).sort_values(by="total", ascending=False).drop(columns="total").head(15))

Tabla dinámica de registros por cultivo y año (Top 15 por registros totales):


anio,2023,2024
Cultivo,,
Papa,239.0,237.0
Maíz,194.0,196.0
Arveja,176.0,168.0
Frijol,162.0,169.0
Tomate,113.0,120.0
Cebolla de bulbo,93.0,95.0
Yuca,82.0,82.0
Caña,63.0,66.0
Trigo,58.0,57.0


**Interpretación del resultado:**

Concentración de la frecuencia muestral en tubérculos y leguminosas: La Papa es el cultivo con mayor número de reportes registrados en Boyacá tanto en 2023 ($239$ registros, $9,81\%$) como en 2024 ($248$ registros, $9,97\%$), seguido por el Maíz y la Arveja. Esto refleja la alta fragmentación y dispersión municipal con la que el DANE/EVA realiza la toma de información para estos cultivos insignia del departamento.

Estabilidad temporal del volumen de reporte municipal: La estructura de observaciones se mantiene sumamente homogénea entre ambos años (2.436 registros en 2023 frente a 2.488 en 2024). La presencia de ceros en la tabla dinámica para ciertos cultivos secundarios indica que no se capturaron observaciones de producción cosechada para esa combinación específica en el municipio y año correspondientes.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.